In [18]:
import re
import nltk
import joblib
import numpy as np
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

lemmatizer = nltk.stem.WordNetLemmatizer()

MODEL_PATH = "sentiment_classifier.joblib"
LABELS = {0 : "negative", 1: "positive"}

def text_processor(text: str) -> str:
    text = text.lower()

    # HTML Tags, URLs, Puncuation and Special Chars
    text = re.sub(r"<.+?>", "", text) # HTML TAGS
    text = re.sub(r"[(http(s)?):\/\/(www\.)?a-zA-Z0-9@:%._\+~#=]{2,256}\.[a-z]{2,6}\b([-a-zA-Z0-9@:%_\+.~#?&//=]*)", "", text) # URLS
    text = re.sub(r"[^\w\d\s]", " ", text) # Puncuation or special Chars
    text = text.strip()
    
    # Stopwords
    text = [word for word in text.split() if word not in ENGLISH_STOP_WORDS]
    text = [word for word in text if len(word) > 2]
    text = " ".join(text)

    # Lemmatization
    text = lemmatizer.lemmatize(text)

    return text

def get_model(file_path :str):
    try:
        package = joblib.load(file_path)
        classifier = package["classifier"]
        vectorizer = package["vectorizer"]
    except Exception as e:
        print(f"Cannot load model files , error {e}")
        return None, None

    return classifier, vectorizer

def get_embeddings(tokens, model):
    vectors = [ model[word] for word in tokens if word in model ]
    return np.mean(vectors, axis = 0 )

def prediction_pipeline(
    user_input : str , model_path: str = MODEL_PATH, labels: dict = LABELS
)-> str:
    classifier, vectorizer = get_model(model_path)
    processed_text = text_processor(user_input)
    
    embedddings = np.array([
        get_embeddings(processed_text.split(), vectorizer)
    ])
    predicted_sentiment = classifier.predict(embedddings)

    return labels[predicted_sentiment[0]].capitalize()

In [ ]:
import re
import nltk
import joblib
import numpy as np
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

MODEL_PATH = "sentiment_classifier.joblib"
LABELS = {0: "negative",1: "positive"}

class SentimentPredictor:
    def __init__(self, model_path=None, labels=None):
        self.model_path = model_path
        self.labels = labels
        self.lemmatizer = nltk.stem.WordNetLemmatizer()

        self.classifier = None
        self.vectorizer = None

        self.load_model()

    def text_processor(self, text: str) -> str:
        """Clean and preprocess input text."""

        text = text.lower()
        # HTML Tags
        text = re.sub(r"<.+?>", "", text)
        # URLs
        text = re.sub(r"[(http(s)?):\/\/(www\.)?a-zA-Z0-9@:%._\+~#=]{2,256}\.[a-z]{2,6}\b([-a-zA-Z0-9@:%_\+.~#?&//=]*)",text)
        # Punctuation and special characters
        text = re.sub(r"[^\w\d\s]", " ", text)
        text = text.strip()
        # Stopwords
        words = [word for word in text.split() if word not in ENGLISH_STOP_WORDS]
        # Remove short words
        words = [word for word in words if len(word) > 2]
        text = " ".join(words)

        # Lemmatization
        text = self.lemmatizer.lemmatize(text)

        return text

    def load_model(self):
        """Load classifier and vectorizer from the joblib file."""

        try:
            package = joblib.load(self.model_path)

            self.classifier = package["classifier"]
            self.vectorizer = package["vectorizer"]

        except Exception as e:
            raise RuntimeError(f"Cannot load model files: {e}") from e

    def get_embeddings(self, tokens):
        """Generate the mean embedding for the given tokens."""

        vectors = [
            self.vectorizer[word] for word in tokens if word in self.vectorizer
        ]

        if not vectors:
            raise ValueError("No valid words found in the vectorizer.")

        return np.mean(vectors, axis=0)

    def predict(self, user_input: str) -> str:
        """Predict sentiment for the given input."""

        processed_text = self.text_processor(user_input)

        embeddings = np.array([
            self.get_embeddings(processed_text.split())
        ])

        predicted_sentiment = self.classifier.predict(embeddings)

        return self.labels[predicted_sentiment[0]].capitalize()

    def run(self, user_input : str):
        """Main application entry point."""
        try:
            sentiment = self.predict(user_input)
            print(f"Predicted Sentiment: {sentiment}")

        except Exception as e:
            print(f"Prediction failed: {e}")


if __name__ == "__main__":
    predictor = SentimentPredictor()
    predictor.run()


In [19]:
user_input = "Phil the Alien is one of those quirky films where the humour is based around the oddness of everything rather than actual punchlines.<br /><br />At first it was very odd and pretty funny but as the movie progressed I didn\'t find the jokes or oddness funny anymore.<br /><br />Its a low budget film (thats never a problem in itself), there were some pretty interesting characters, but eventually I just lost interest.<br /><br />I imagine this film would appeal to a stoner who is currently partaking.<br /><br />For something similar but better try 'Brother from another planet'"

In [20]:
prediction_pipeline( user_input )

'Negative'